# 🚀 L&D Designs — Automated Bulk Sender
Automatically sends **emails** (Gmail) and **SMS** (Twilio) to all 2000 leads in one go.

**Steps:**
1. Cell 1 — Install packages
2. Cell 2 — Enter your credentials
3. Cell 3 — Upload your leads file
4. Cell 4 — Preview & confirm
5. Cell 5 — Send everything
6. Cell 6 — Download log

> **SMS costs money via Twilio** (~£0.04/message). Leave `TWILIO_ACCOUNT_SID` blank to skip SMS and only send emails (free).

In [ ]:
# ── Cell 1: Install packages ───────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl', 'twilio'])
print('Ready.')

---
## Cell 2 — Your credentials

### Gmail setup (for emails — free)
1. Go to [myaccount.google.com](https://myaccount.google.com) → Security → 2-Step Verification (enable it)
2. Then go to Security → **App passwords**
3. Type `Lead Outreach` → Create → copy the 16-character password

### Twilio setup (for SMS — paid ~£0.04/text)
1. Sign up at [twilio.com](https://twilio.com)
2. Get a UK phone number
3. Copy your **Account SID** and **Auth Token** from the Twilio Console
4. Leave `TWILIO_ACCOUNT_SID = ''` to skip SMS entirely

In [ ]:
# ── Cell 2: Credentials ────────────────────────────────────────────────────

# ── GMAIL (for emails) ─────────────────────────────────────────────────────
GMAIL_ADDRESS      = 'your@gmail.com'
GMAIL_APP_PASSWORD = 'xxxx xxxx xxxx xxxx'   # 16-char app password

# ── TWILIO (for SMS — leave blank to skip) ─────────────────────────────────
TWILIO_ACCOUNT_SID = ''                      # e.g. 'ACxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
TWILIO_AUTH_TOKEN  = ''                      # e.g. 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
TWILIO_FROM_NUMBER = ''                      # e.g. '+441234567890'

# ── YOUR PRICING PAGE ──────────────────────────────────────────────────────
PRICING_LINK = 'https://mellow-speculoos-d1850c.netlify.app/pricing.html'

# ── SETTINGS ───────────────────────────────────────────────────────────────
EMAIL_DELAY_SECONDS = (3, 6)   # random pause between emails (min, max)
SMS_DELAY_SECONDS   = (1, 3)   # random pause between SMS messages
SEND_EMAILS = True             # set False to skip emails
SEND_SMS    = bool(TWILIO_ACCOUNT_SID)  # auto-disabled if no Twilio creds

print('Gmail :', GMAIL_ADDRESS)
print('Emails:', 'ENABLED' if SEND_EMAILS else 'DISABLED')
print('SMS   :', 'ENABLED via Twilio' if SEND_SMS else 'DISABLED (no Twilio credentials)')

In [ ]:
# ── Cell 3: Upload leads file ──────────────────────────────────────────────
from google.colab import files
import openpyxl
import re

print('Select your leads .xlsx file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

wb = openpyxl.load_workbook(filename)
ws = wb.active
headers = [str(cell.value or '').strip() for cell in ws[1]]
print('Columns found:', headers)

all_leads = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if any(v is not None for v in row):
        all_leads.append(dict(zip(headers, row)))

print('Total rows loaded:', len(all_leads))

# ── Helpers ────────────────────────────────────────────────────────────────
OWN_DOMAIN = 'lddesigns'

def get_field(lead, *keys):
    for k in keys:
        v = lead.get(k)
        if v and str(v).strip() and str(v).strip().lower() not in ('none','nan','null',''):
            return str(v).strip()
    return ''

def clean_phone(phone):
    d = re.sub(r'[^\d+]', '', phone)
    if d.startswith('0'):
        d = '+44' + d[1:]
    elif d and not d.startswith('+'):
        d = '+44' + d
    return d

def is_mobile_uk(phone):
    d = re.sub(r'[^\d]', '', phone)
    if d.startswith('447'): return True
    return d.startswith('07')

# ── Filter leads ───────────────────────────────────────────────────────────
email_leads = []
sms_leads   = []
skip_website = skip_no_contact = skip_own = 0

for lead in all_leads:
    status  = str(get_field(lead, 'Website Status', 'Status') or '').upper()
    email   = get_field(lead, 'Email', 'email', 'Email Address')
    phone   = get_field(lead, 'Phone', 'phone', 'Phone Number')

    if 'ACTIVE' in status:
        skip_website += 1
        continue

    if email and '@' in email and OWN_DOMAIN.lower() not in email.lower():
        email_leads.append(lead)
    elif not email:
        skip_no_contact += 1

    if phone and is_mobile_uk(phone):
        sms_leads.append(lead)

print()
print('── Filter Results ───────────────────')
print('Skipped (has website)  :', skip_website)
print('Skipped (no contact)   :', skip_no_contact)
print('Ready to EMAIL         :', len(email_leads))
print('Ready to SMS           :', len(sms_leads))
print()
if email_leads:
    print('First 3 email leads:')
    for l in email_leads[:3]:
        print(' -', get_field(l,'Business Name','name'), '|', get_field(l,'Email','email','Email Address'))
if sms_leads:
    print('First 3 SMS leads:')
    for l in sms_leads[:3]:
        print(' -', get_field(l,'Business Name','name'), '|', get_field(l,'Phone','phone','Phone Number'))

In [ ]:
# ── Cell 4: Preview messages ───────────────────────────────────────────────

EMAIL_SUBJECT = 'Quick question — website for {name}'

EMAIL_BODY = '''Hi,

I noticed {name} doesn't have a website yet.

I build professional websites for local businesses in Wigan from just £199 — usually done within a week. No monthly fees, one-off payment.

Here's an example of my work: ''' + PRICING_LINK + '''

Would you be interested in a free quote? Just reply to this email.

Thanks,
Dylan
L&D Designs
07301 181878'''

SMS_MESSAGE = ("Hi, I noticed {name} doesn't have a website. "
               "I build professional sites for Wigan businesses from £199 — "
               "done in days, no monthly fees. Free quote? "
               "Reply or call — Dylan, L&D Designs")

# Preview
sample_name = get_field(email_leads[0], 'Business Name', 'name') if email_leads else 'Sample Business'

print('=' * 60)
print('EMAIL PREVIEW')
print('=' * 60)
print('Subject:', EMAIL_SUBJECT.format(name=sample_name))
print('-' * 60)
print(EMAIL_BODY.format(name=sample_name))
print()
print('=' * 60)
print('SMS PREVIEW')
print('=' * 60)
print(SMS_MESSAGE.format(name=sample_name))
print('(' + str(len(SMS_MESSAGE.format(name=sample_name))) + ' chars)')
print()
print('PLAN:')
print(' Emails to send :', len(email_leads) if SEND_EMAILS else 0)
print(' SMS to send    :', len(sms_leads)   if SEND_SMS   else 0, '(Twilio)' if SEND_SMS else '(disabled)')

In [ ]:
# ── Cell 5: Send everything ────────────────────────────────────────────────
import smtplib, time, random, csv
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Safety check
if GMAIL_ADDRESS == 'your@gmail.com':
    raise ValueError('Fill in your Gmail address in Cell 2 first.')
if GMAIL_APP_PASSWORD == 'xxxx xxxx xxxx xxxx':
    raise ValueError('Fill in your Gmail App Password in Cell 2 first.')

confirm = input(
    f'About to send {len(email_leads) if SEND_EMAILS else 0} emails '
    f'and {len(sms_leads) if SEND_SMS else 0} SMS messages. '
    'Type YES to confirm: '
).strip()

if confirm != 'YES':
    print('Cancelled. Nothing was sent.')
else:
    send_log = []
    email_sent = email_failed = sms_sent = sms_failed = 0

    # ── EMAIL BATCH ────────────────────────────────────────────────────────
    if SEND_EMAILS and email_leads:
        print()
        print('Connecting to Gmail SMTP...')
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        print('Connected. Sending emails...')
        print('-' * 50)

        for i, lead in enumerate(email_leads):
            name  = get_field(lead, 'Business Name', 'name') or 'there'
            email = get_field(lead, 'Email', 'email', 'Email Address')
            ts    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            try:
                msg = MIMEMultipart()
                msg['From']    = GMAIL_ADDRESS
                msg['To']      = email
                msg['Subject'] = EMAIL_SUBJECT.format(name=name)
                msg.attach(MIMEText(EMAIL_BODY.format(name=name), 'plain'))
                server.sendmail(GMAIL_ADDRESS, email, msg.as_string())
                email_sent += 1
                print(f'[{i+1}/{len(email_leads)}] Sent -> {name} | {email}')
                send_log.append({'type':'email','business':name,'contact':email,'status':'sent','time':ts})
            except Exception as e:
                email_failed += 1
                print(f'[{i+1}/{len(email_leads)}] FAILED -> {name} | {email} | {e}')
                send_log.append({'type':'email','business':name,'contact':email,'status':f'failed: {e}','time':ts})

            if i < len(email_leads) - 1:
                time.sleep(random.uniform(*EMAIL_DELAY_SECONDS))

        server.quit()
        print(f'\nEmails done. Sent: {email_sent}  Failed: {email_failed}')

    # ── SMS BATCH ──────────────────────────────────────────────────────────
    if SEND_SMS and sms_leads:
        from twilio.rest import Client
        twilio_client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
        print()
        print('Sending SMS messages via Twilio...')
        print('-' * 50)

        for i, lead in enumerate(sms_leads):
            name  = get_field(lead, 'Business Name', 'name') or 'there'
            phone = get_field(lead, 'Phone', 'phone', 'Phone Number')
            phone_e164 = clean_phone(phone)
            ts    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            try:
                twilio_client.messages.create(
                    body=SMS_MESSAGE.format(name=name),
                    from_=TWILIO_FROM_NUMBER,
                    to=phone_e164
                )
                sms_sent += 1
                print(f'[{i+1}/{len(sms_leads)}] SMS sent -> {name} | {phone_e164}')
                send_log.append({'type':'sms','business':name,'contact':phone_e164,'status':'sent','time':ts})
            except Exception as e:
                sms_failed += 1
                print(f'[{i+1}/{len(sms_leads)}] SMS FAILED -> {name} | {phone_e164} | {e}')
                send_log.append({'type':'sms','business':name,'contact':phone_e164,'status':f'failed: {e}','time':ts})

            if i < len(sms_leads) - 1:
                time.sleep(random.uniform(*SMS_DELAY_SECONDS))

        print(f'\nSMS done. Sent: {sms_sent}  Failed: {sms_failed}')

    # ── Store log for Cell 6 ───────────────────────────────────────────────
    SEND_LOG = send_log
    print()
    print('=' * 50)
    print('ALL DONE')
    print(f'  Emails sent   : {email_sent}')
    print(f'  Emails failed : {email_failed}')
    print(f'  SMS sent      : {sms_sent}')
    print(f'  SMS failed    : {sms_failed}')
    print('='*50)
    print('Run Cell 6 to download your send log.')

In [ ]:
# ── Cell 6: Download send log ──────────────────────────────────────────────
import csv
from google.colab import files
from datetime import datetime

try:
    SEND_LOG
except NameError:
    print('No log found — run Cell 5 first.')
    raise SystemExit

log_file = 'send_log_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.csv'

with open(log_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['type','business','contact','status','time'])
    writer.writeheader()
    writer.writerows(SEND_LOG)

print('Log saved:', log_file)
files.download(log_file)

# Print failed entries for quick review
failed = [r for r in SEND_LOG if r['status'].startswith('failed')]
if failed:
    print()
    print('Failed entries:')
    for r in failed:
        print(' -', r['type'], '|', r['business'], '|', r['contact'], '|', r['status'])